<a href="https://colab.research.google.com/github/balabhadrabagh/Claude-tool-calling-fundamentals/blob/main/Agentic_Sales_Data_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anthropic pandas openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.1 MB/s eta 0:00:00


In [ ]:
import os
import json
import pandas as pd
from anthropic import Anthropic
from google.colab import userdata

# SET UP ANTHROPIC CLIENT USING COLAB SECRETS
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    client = Anthropic()
except Exception as e:
    print("Error: Could not retrieve ANTHROPIC_API_KEY from Colab secrets. Please ensure it is added with exact name.")

In [ ]:
# Define your Excel file path
EXCEL_FILE_PATH = "/content/Sales_data.xlsx"

# Initialize global conversation memory array
# This stores dictionaries with roles ('user', 'assistant') and content
conversation_history = []

In [2]:
# DATA LOOKUP FUNCTION
def query_sales_data(item_code=None, target_month=None, customer_type=None, time_granularity="month"):
    """
    Loads and filters the excel sheet dynamically based on agent parameters.
    Re-reads the file on each call to handle live changes.
    """
    if not os.path.exists(EXCEL_FILE_PATH):
        return f"Error: '{EXCEL_FILE_PATH}' not found. Check your file path."

    try:
        df = pd.read_excel(EXCEL_FILE_PATH)

        # Clean up column headers (lowercase, strip trailing spaces, replace spaces with underscores)
        df.columns = [col.strip().lower().replace(" ", "_").replace("#", "num").replace("(", "").replace(")", "") for col in df.columns]

        df['fulfillment_date'] = pd.to_datetime(df['fulfillment_date'])

        # Ensure item numbers match as strings to preserve leading zeros
        df['item_num'] = df['item_num'].astype(str).str.split('.').str[0]

        # Filter for data starting from Feb 1, 2026 onwards
        df = df[df['fulfillment_date'] >= '2026-02-01']
        df['month'] = df['fulfillment_date'].dt.strftime('%B %Y')

    except Exception as e:
        return f"Data load error: {str(e)}"

    filtered_df = df.copy()

    # Apply dynamic filters
    if item_code:
        filtered_df = filtered_df[filtered_df['item_num'] == str(item_code)]
    if customer_type:
        filtered_df = filtered_df[filtered_df['customer_type'].str.lower().str.contains(customer_type.lower())]
    if target_month:
        filtered_df = filtered_df[filtered_df['month'].str.lower().str.contains(target_month.lower())]

    if filtered_df.empty:
        return "No sales data found matching those criteria from Feb 1, 2026 onwards."

    # Group data based on requested granularity
    if time_granularity == "date":
        filtered_df['time_period'] = filtered_df['fulfillment_date'].dt.strftime('%Y-%m-%d')
    elif time_granularity == "week":
        filtered_df['time_period'] = filtered_df['fulfillment_date'].dt.strftime('%Y-W%U')
    else:
        filtered_df['time_period'] = filtered_df['month']

    # Include customer type in breakdown if relevant
    group_cols = ['item_num', 'item_description', 'time_period']
    if customer_type or len(filtered_df['customer_type'].unique()) > 1:
        group_cols.insert(0, 'customer_type')

    summary = filtered_df.groupby(group_cols)['sales_units_shipped'].sum().reset_index()
    summary = summary.sort_values(by='time_period')

    return summary.to_string(index=False)

In [3]:
# ANTHROPIC TOOL SPECIFICATION
tools_config = [
    {
        "name": "query_sales_data",
        "description": "Queries sales quantities from the local ledger starting Feb 1, 2026. Use this to filter by date, week, month, customer types, or specific item codes.",
        "input_schema": {
            "type": "object",
            "properties": {
                "item_code": {
                    "type": "string",
                    "description": "The SKU or item code number (e.g., '9608871886')."
                },
                "target_month": {
                    "type": "string",
                    "description": "The month name to filter by (e.g., 'February', 'March')."
                },
                "customer_type": {
                    "type": "string",
                    "description": "The type of customer account (e.g., 'Resorts', 'Wholesale Boutiques')."
                },
                "time_granularity": {
                    "type": "string",
                    "enum": ["month", "week", "date"],
                    "description": "Time grouping style. Use 'date' for exact days, 'week' for weekly rollups, or 'month' for generic monthly summaries. Defaults to 'month'."
                }
            }
        }
    }
]


In [1]:
def run_chat_agent(user_message):
    global conversation_history

    # Using the standard production identifier
    model = "claude-sonnet-5"

    conversation_history.append({"role": "user", "content": user_message})

    # Keep the last 5 turns to prevent token bloat
    if len(conversation_history) > 10:
        conversation_history = conversation_history[-10:]

    try:
        response = client.messages.create(
            model=model,
            max_tokens=1000,
            tools=tools_config,
            messages=conversation_history
        )
    except Exception as e:
        print(f"Anthropic API call failed: {e}")
        return "Sorry, I hit an API error. Please try again."

    # Check if the model requested a tool call
    if response.stop_reason == "tool_use":
        tool_call = next(b for b in response.content if b.type == "tool_use")

        print(f"LOG: Agent calling tool '{tool_call.name}' with args: {tool_call.input}")

        if tool_call.name == "query_sales_data":
            tool_result = query_sales_data(**tool_call.input)
        else:
            tool_result = f"Error: Tool {tool_call.name} not recognized."

        # Append tool interactions to context history
        conversation_history.append({"role": "assistant", "content": response.content})
        conversation_history.append({
            "role": "user",
            "content": [{"type": "tool_result", "tool_use_id": tool_call.id, "content": tool_result}]
        })

        # Get final answer from model after tool executes
        final_response = client.messages.create(
            model=model,
            max_tokens=1000,
            tools=tools_config,
            messages=conversation_history
        )

        if isinstance(final_response.content, list):
            final_text = "".join([b.text for b in final_response.content if hasattr(b, 'text')])
        else:
            final_text = final_response.content.text if hasattr(final_response.content, 'text') else str(final_response.content)

        # Clean up history by replacing intermediate logs with final answer string
        conversation_history = conversation_history[:-2]
        conversation_history.append({"role": "assistant", "content": final_text})
        return final_text

    else:
        # Standard text reply without tool use
        if isinstance(response.content, list):
            reply = "".join([b.text for b in response.content if hasattr(b, 'text')])
        else:
            reply = response.content.text if hasattr(response.content, 'text') else str(response.content)

        conversation_history.append({"role": "assistant", "content": reply})
        return reply

In [ ]:
print(run_chat_agent("Which month did SKU 9608871886 ship the most units?"))

**February** remains the month with the highest shipment volume for SKU 9608871886, with **119 total units** shipped.

Quick recap of the monthly totals:

| Month | Total Units Shipped |
|-----------|---------------------|
| **February** | **119** |
| June | 84 |
| July | 38 |
| May | 32 |
| April | 25 |
| March | 15 |
| August | 14 |

Let me know if you'd like me to dig deeper into February's breakdown by customer type, or pull daily/weekly data within that month!


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Create the visual elements for the real-time chat window
output_area = widgets.Output(layout=widgets.Layout(height='400px', overflow_y='auto', border='1px solid #ddd', padding='10px', margin='0 0 10px 0'))
text_input = widgets.Text(placeholder='Type your sales question here...', layout=widgets.Layout(width='80%'))
send_button = widgets.Button(description='Send', button_style='primary', layout=widgets.Layout(width='18%'))

# Combine the text box and send button side-by-side
input_box = widgets.HBox([text_input, send_button])

def display_message(sender, text, is_bot=False):
    """Renders messages cleanly using HTML formatting for a realistic chat bubble style."""
    align = "left" if is_bot else "right"
    bg_color = "#f1f0f0" if is_bot else "#007bff"
    text_color = "#333" if is_bot else "#fff"

    message_html = f"""
    <div style="text-align: {align}; margin: 8px 0;">
        <div style="display: inline-block; background-color: {bg_color}; color: {text_color};
                    padding: 8px 14px; border-radius: 12px; max-width: 75%; text-align: left;
                    font-family: Arial, sans-serif; font-size: 14px; white-space: pre-wrap;">
            <b>{"Bot" if is_bot else "You"}:</b> {text}
        </div>
    </div>
    """
    with output_area:
        display(HTML(message_html))

def on_send_clicked(b):
    user_query = text_input.value.strip()
    if not user_query:
        return

    # Clear input field immediately for crisp user experience
    text_input.value = ''

    # Render user's message in the chat pane
    display_message("You", user_query, is_bot=False)

    # Show loading status message
    with output_area:
        status_id = display(HTML("<i style='color: #888; font-size: 13px;'>Agent thinking...</i>"), display_id=True)

    try:
        # Pass input into your existing backend agent state loop
        bot_response = run_chat_agent(user_query)

        # Remove the loading message and display final answer
        status_id.update(HTML(""))
        display_message("Bot", bot_response, is_bot=True)

    except Exception as e:
        status_id.update(HTML(f"<b style='color: red;'>System Error: {str(e)}</b>"))

# Bind the trigger events to both clicking the button and pressing Enter
send_button.on_click(on_send_clicked)
text_input.on_submit(on_send_clicked)

# Render the widget component view onto the screen
print("=== LIVE SALES CONVERSATION AGENT ===")
display(widgets.VBox([output_area, input_box]))


=== LIVE SALES CONVERSATION AGENT ===
